## 🚕 NYC YELLOW TAXI ENRICHMENT PIPELINE

---

### 📥 **Step 1: Read Data from Source Tables**

> In this stage, we read cleansed yellow taxi trip data and taxi zone lookup information from the Databricks tables.

---

In [0]:
from datetime import datetime

# =====================================================
# CALCULATE START TIME
# =====================================================

load_start_time = datetime.now()

In [0]:
df_trip_cleansed = spark.read.table("NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED")

In [0]:
df_zones = spark.read.table("NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP")

#### JOIN TRIPS WITH PICKUP ZONE DETAILS TO OBTAIN BOROUGH AND ZONE NAME
- NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED
- NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP

In [0]:
df_taxi_cleasned_df_lookup = df_trip_cleansed.join(
                df_zones, 
                df_trip_cleansed.pu_location_id == df_zones.location_id,
                "left"
                ).select(
                    df_trip_cleansed.vendor,
                    df_trip_cleansed.tpep_pickup_datetime,
                    df_trip_cleansed.tpep_dropoff_datetime,
                    df_trip_cleansed.trip_duration,
                    df_trip_cleansed.passenger_count,
                    df_trip_cleansed.trip_distance,
                    df_trip_cleansed.rate_type,
                    df_zones.borough.alias("pu_borough"),   # pickup borough
                    df_zones.zone.alias("pu_zone"),         # pickup zone
                    df_trip_cleansed.do_location_id,                # dropoff location ID for next join
                    df_trip_cleansed.payment_type,
                    df_trip_cleansed.fare_amount,
                    df_trip_cleansed.extra,
                    df_trip_cleansed.mta_tax,
                    df_trip_cleansed.tolls_amount,
                    df_trip_cleansed.improvement_surcharge,
                    df_trip_cleansed.total_amount,
                    df_trip_cleansed.congestion_surcharge,
                    df_trip_cleansed.airport_fee,  
                    df_trip_cleansed.cbd_congestion_fee,
                    df_trip_cleansed.load_timestamp
                )

#### JOIN TRIPS WITH PICKUP ZONE DETAILS TO OBTAIN ENRICHMENT
- NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED
- NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP

In [0]:
df_cleansed_lookup_trans = df_taxi_cleasned_df_lookup.join(
                                df_zones, 
                                df_taxi_cleasned_df_lookup.do_location_id == df_zones.location_id,
                                "left"
                                ).select(
                                            df_taxi_cleasned_df_lookup.vendor,
                                            df_taxi_cleasned_df_lookup.tpep_pickup_datetime,
                                            df_taxi_cleasned_df_lookup.tpep_dropoff_datetime,
                                            df_trip_cleansed.trip_duration,
                                            df_taxi_cleasned_df_lookup.passenger_count,
                                            df_taxi_cleasned_df_lookup.trip_distance,
                                            df_taxi_cleasned_df_lookup.rate_type,
                                            df_taxi_cleasned_df_lookup.pu_borough,
                                            df_zones.borough.alias("do_borough"), # dropoff borough
                                            df_taxi_cleasned_df_lookup.pu_zone,
                                            df_zones.zone.alias("do_zone"),       # dropoff zone
                                            df_taxi_cleasned_df_lookup.payment_type,
                                            df_taxi_cleasned_df_lookup.fare_amount,
                                            df_taxi_cleasned_df_lookup.extra,
                                            df_taxi_cleasned_df_lookup.mta_tax,
                                            df_taxi_cleasned_df_lookup.tolls_amount,
                                            df_taxi_cleasned_df_lookup.improvement_surcharge,
                                            df_taxi_cleasned_df_lookup.total_amount,
                                            df_taxi_cleasned_df_lookup.congestion_surcharge,
                                            df_taxi_cleasned_df_lookup.airport_fee,  
                                            df_taxi_cleasned_df_lookup.cbd_congestion_fee,
                                            df_taxi_cleasned_df_lookup.load_timestamp
                                )

In [0]:
from datetime import datetime

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType,
    DecimalType,
    DateType
)

# =====================================================
# WORKFLOW PARAMETERS
# =====================================================

dbutils.widgets.text("log_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("event_time", "")
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("layer", "")
dbutils.widgets.text("notebook_path", "")
dbutils.widgets.text("pipeline_name", "")

# =====================================================
# RETRIEVE PARAMETERS
# =====================================================

log_id = dbutils.widgets.get("log_id")
run_id = dbutils.widgets.get("run_id")
raw_event_time = dbutils.widgets.get("event_time")

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
layer = dbutils.widgets.get("layer")

notebook_path = dbutils.widgets.get("notebook_path")
pipeline_name = dbutils.widgets.get("pipeline_name")

# =====================================================
# EVENT TIME
# =====================================================

try:
    if not raw_event_time or raw_event_time.startswith("{{"):
        event_time = datetime.now()
    else:
        event_time = datetime.fromisoformat(raw_event_time)
except Exception:
    event_time = datetime.now()

# =====================================================
# CURRENT USER
# =====================================================

user_name = spark.sql("SELECT current_user() AS user_name").first()["user_name"]

# =====================================================
# INITIALIZE AUDIT VARIABLES
# =====================================================

record_count = 0
status = "FAILED"
event_type = "LOAD_FAILURE"
message = ""

# =====================================================
# BUSINESS LOAD
# =====================================================

try:

    record_count = df_cleansed_lookup_trans.count()

    # Target Write
    df_cleansed_lookup_trans.write.mode('overwrite').saveAsTable('NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_ENRICHED')

    status = "SUCCESS"
    event_type = "LOAD_SUCCESS"
    message = f"Loaded {record_count} records into {target_table}"

except Exception as e:

    status = "FAILED"
    event_type = "LOAD_FAILURE"
    message = str(e)

# =====================================================
# CAPTURE END TIME
# =====================================================

load_end_time = datetime.now()

# =====================================================
# AUDIT SCHEMA
# =====================================================

audit_schema = StructType([

    StructField("log_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("event_time", DateType(), True),
    StructField("event_type", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("load_start_time", TimestampType(), True),
    StructField("load_end_time", TimestampType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_extension", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("file_size_bytes", LongType(), True),
    StructField("file_size_mb", DecimalType(18, 2), True),
    StructField("file_created_time", TimestampType(), True),
    StructField("file_modified_time", TimestampType(), True)

])

# =====================================================
# BUILD AUDIT RECORD
# =====================================================

audit_data = [(

    log_id,
    run_id,
    event_time,
    event_type,
    source_table,
    target_table,
    layer,
    record_count,
    status,
    message,
    user_name,
    notebook_path,
    pipeline_name,
    load_start_time,
    load_end_time,
    None,  # file_name
    None,  # file_path
    None,  # file_extension
    None,  # source_system
    None,  # source_folder
    None,  # file_size_bytes
    None,  # file_size_mb
    None,  # file_created_time
    None   # file_modified_time
)]

audit_df = spark.createDataFrame(
    audit_data,
    audit_schema
)

# =====================================================
# WRITE AUDIT LOG
# =====================================================

try:

    audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG")

    print(f"Audit logging completed successfully for Run ID: {run_id}")

except Exception as audit_error:

    print(f"Business load completed but audit logging failed: {audit_error}")

# =====================================================
# FAIL NOTEBOOK IF LOAD FAILED
# =====================================================

if status == "FAILED":
    raise Exception(message)

#### LOAD THE DATA INTO THE SILVER TABLE
- NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_ENRICHED

In [0]:
%skip
df_cleansed_lookup_trans.write.mode("overwrite").saveAsTable("NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_ENRICHED")
df_cleansed_lookup_trans

In [0]:
dbutils.notebook.exit('YELLOW TAXI TRIP HAS BEEN LOADED INTO NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_ENRICHED')